# 04 - Run everything (Runtime > Run all)
Runs, in order: setup check, unit tests, GPU benchmarks (surface code + qLDPC), the NVIDIA Ising head-to-head, the full AI-pre-decoder + GPU-decoder pipeline vs CPU baselines, an optional CUDA-Q QEC step, then builds the website and downloads one zip.

**Before you run:** Runtime > Change runtime type > **T4 GPU**. Settings are in the first code cell. Start with `PROFILE = "quick"`.

**Ising step (optional):** accept the terms at https://huggingface.co/nvidia/ising_decoder_surface_code_1_fast , create a *read* token and add it as the Colab secret `HF_TOKEN` (key icon, notebook access ON). Without it that step is skipped and everything else still runs. Never paste the token into a cell.

In [ ]:
# ---- settings ----
PROFILE = "quick"   # "quick" (~5 min) first, then "full" (~20-40 min)
RUN_ISING = True    # needs the Colab secret HF_TOKEN (see the markdown cell above); skipped if missing
RUN_CUDAQ = False   # optional: pip-installs cudaq-qec, which can change the environment
GITHUB_REPO = "Rhytam23/ccn"
import os, subprocess, sys
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{GITHUB_REPO}.git", "/content/repo"], check=True)
%cd /content/repo
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps   # for shell commands (pytest, scripts)
sys.path.insert(0, "/content/repo/src")  # for this kernel: editable installs are not visible to an already-running kernel
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
!pip -q install pytest
!pytest -q

In [ ]:
!python scripts/run_benchmarks.py --profile $PROFILE --device cuda --tag colab-gpu

In [ ]:
import os, subprocess
if RUN_ISING:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # never printed
    except Exception as exc:
        print("Skipping Ising head-to-head: no HF_TOKEN secret found (", type(exc).__name__, ")")
        RUN_ISING = False
if RUN_ISING:
    if not os.path.exists("third_party/Ising-Decoding"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/NVIDIA/Ising-Decoding.git", "third_party/Ising-Decoding"], check=True)
    subprocess.run("pip -q install safetensors omegaconf hydra-core huggingface_hub beliefmatching", shell=True)
    subprocess.run("python scripts/run_ising_bench.py --repo third_party/Ising-Decoding --download --device cuda --distances 9 13 --ps 0.003 0.005 --shots 20000 --tag colab-ising", shell=True)
    subprocess.run("python scripts/run_full_pipeline.py --repo third_party/Ising-Decoding --download --device cuda --distances 9 13 --ps 0.003 0.005 --shots 20000 --cpu-shots 1000 --tag colab-pipeline", shell=True)

In [ ]:
import subprocess
if RUN_CUDAQ:
    subprocess.run("pip -q install cudaq-qec", shell=True)
    from qechybrid import cudaq_qec_adapter
    print("cudaq_qec:", cudaq_qec_adapter.available())
    subprocess.run("python scripts/run_benchmarks.py --profile quick --device cuda --tag colab-qldpc-cudaq --only qldpc", shell=True)
else:
    print("CUDA-Q QEC step skipped (RUN_CUDAQ = False)")

In [ ]:
!python scripts/make_report.py
!zip -qr results_colab.zip results docs/index.html docs/figs

In [ ]:
import glob, pandas as pd
for f in sorted(glob.glob("results/colab-*/*.csv")):
    df = pd.read_csv(f)
    print("\n==", f, len(df), "rows")
    cols = [c for c in ["code", "d", "p", "decoder", "ler", "throughput_sps", "accept_rate"] if c in df.columns]
    print(df[cols].to_string(index=False, float_format=lambda x: f"{x:.3g}"))

In [ ]:
from google.colab import files
files.download('results_colab.zip')